# Cliff Walking: Sarsa, Q-learning, and Expected Sarsa Comparison

This notebook provides a detailed theoretical explanation and Python implementation comparing three temporal-difference (TD) control methods—**Sarsa (on-policy)**, **Q-learning (off-policy)**, and **Expected Sarsa (on/off-policy)**—on the classic **Cliff Walking Gridworld** (Example 6.6, Sutton & Barto).

## 1. Theoretical Background & Backup Diagrams

Temporal Difference (TD) control methods update action-value estimates $Q(S_t, A_t)$ based on transitions from state to state. The main difference lies in how they estimate the value of the next state $\Phi(S_{t+1})$:

*   **Sarsa**: Updates towards the value of the action actually chosen by the behavior policy: $Q(S_{t+1}, A_{t+1})$.
*   **Q-learning**: Updates towards the maximum value of the next state: $\max_a Q(S_{t+1}, a)$, assuming a greedy target policy.
*   **Expected Sarsa**: Updates towards the expected value under the target policy: $\sum_a \pi(a \mid S_{t+1}) Q(S_{t+1}, a)$.

### Backup Diagrams
The update patterns are visually summarized in the following backup diagrams:

| Sarsa | Q-learning | Expected Sarsa |
|:---:|:---:|:---:|
| ![Sarsa Backup](./assets/diagrams/sarsa_backup.svg) | ![Q-learning Backup](./assets/diagrams/q_learning_backup.svg) | ![Expected Sarsa Backup](./assets/diagrams/expected_sarsa_backup.svg) |

## 2. Update Rules at a Glance

### Sarsa (On-policy)
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma\, Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \right]$$

### Q-learning (Off-policy)
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t) \right]$$

### Expected Sarsa (On/Off-policy)
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \sum_a \pi(a \mid S_{t+1})\, Q(S_{t+1}, a) - Q(S_t, A_t) \right]$$

In [ ]:
# Imports and environment configuration
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

# Set random seed for reproducibility
np.random.seed(42)

## 3. Environment & Agent Implementations
We use the Gymnasium `CliffWalking-v1` environment. The environment consists of a $4 \times 12$ gridworld:
* **Start state `S`**: Bottom-left corner $(3, 0)$ or state index 36.
* **Goal state `G`**: Bottom-right corner $(3, 11)$ or state index 47.
* **Cliff `C`**: Bottom row indices $(3, 1)$ to $(3, 10)$ (states 37-46). Stepping into the cliff yields a reward of $-100$ and resets the agent to `S`.
* **Normal Step**: Any other transition yields a reward of $-1$.

In [ ]:
class TDAgent:
    def __init__(self, num_states, num_actions, alpha=0.1, gamma=1.0, epsilon=0.1):
        self.num_states = num_states
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = np.zeros((num_states, num_actions))
        
    def choose_action(self, state):
        # Epsilon-greedy behavior policy
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)
        else:
            q_values = self.Q[state]
            max_q = np.max(q_values)
            best_actions = np.flatnonzero(q_values == max_q)
            if len(best_actions) == 1:
                return best_actions[0]
            return np.random.choice(best_actions)
            
    def get_action_probabilities(self, state):
        # Returns target policy probabilities (epsilon-greedy)
        q_values = self.Q[state]
        max_q = np.max(q_values)
        best_actions = np.flatnonzero(q_values == max_q)
        num_best = len(best_actions)
        
        probs = np.full(self.num_actions, self.epsilon / self.num_actions)
        probs[best_actions] += (1.0 - self.epsilon) / num_best
        return probs
        
    def update_sarsa(self, s, a, r, ns, na):
        # Sarsa update
        td_target = r + self.gamma * self.Q[ns, na]
        self.Q[s, a] += self.alpha * (td_target - self.Q[s, a])
        
    def update_q_learning(self, s, a, r, ns):
        # Q-learning update
        max_q_ns = np.max(self.Q[ns])
        td_target = r + self.gamma * max_q_ns
        self.Q[s, a] += self.alpha * (td_target - self.Q[s, a])
        
    def update_expected_sarsa(self, s, a, r, ns, terminated):
        # Expected Sarsa update
        if terminated:
            expected_val = 0.0
        else:
            probs = self.get_action_probabilities(ns)
            expected_val = np.sum(probs * self.Q[ns])
            
        td_target = r + self.gamma * expected_val
        self.Q[s, a] += self.alpha * (td_target - self.Q[s, a])

In [ ]:
def run_episode(agent, env, method):
    state, info = env.reset()
    total_reward = 0.0
    done = False
    steps = 0
    max_steps = 150  # Cap to prevent excessive looping in early learning
    
    if method == 'sarsa':
        action = agent.choose_action(state)
        while not done and steps < max_steps:
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            next_action = agent.choose_action(next_state)
            agent.update_sarsa(state, action, reward, next_state, next_action)
            
            state = next_state
            action = next_action
            total_reward += reward
            steps += 1
            
    elif method == 'q_learning':
        while not done and steps < max_steps:
            action = agent.choose_action(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            agent.update_q_learning(state, action, reward, next_state)
            
            state = next_state
            total_reward += reward
            steps += 1
            
    elif method == 'expected_sarsa':
        while not done and steps < max_steps:
            action = agent.choose_action(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            agent.update_expected_sarsa(state, action, reward, next_state, terminated)
            
            state = next_state
            total_reward += reward
            steps += 1
            
    return total_reward

## 4. Simulation & Performance Comparison
We run Sarsa, Q-learning, and Expected Sarsa agents for different values of step-size $\alpha \in [0.1, 1.0]$.
We evaluate two performance indicators:
* **Interim Performance**: Average reward per episode over the first 100 episodes.
* **Asymptotic Performance**: Average reward per episode over the last 150 episodes (from episode 101 to 250).

In [ ]:
env = gym.make('CliffWalking-v1')
step_sizes = np.arange(0.1, 1.1, 0.1)
runs = 5
episodes = 250

performance = np.zeros((6, len(step_sizes)))
# 0: Asymptotic Sarsa, 1: Asymptotic Expected Sarsa, 2: Asymptotic Q-learning
# 3: Interim Sarsa,      4: Interim Expected Sarsa,      5: Interim Q-learning

methods = ['sarsa', 'expected_sarsa', 'q_learning']

for m_idx, method in enumerate(methods):
    print(f"Running simulation for {method}...")
    for a_idx, alpha in enumerate(step_sizes):
        run_history = np.zeros((runs, episodes))
        for run in range(runs):
            agent = TDAgent(env.observation_space.n, env.action_space.n, alpha=alpha, gamma=1.0, epsilon=0.1)
            for ep in range(episodes):
                run_history[run, ep] = run_episode(agent, env, method)
                
        performance[m_idx, a_idx] = np.mean(run_history[:, 100:])  # Asymptotic
        performance[m_idx + 3, a_idx] = np.mean(run_history[:, :100])  # Interim
env.close()
print("Simulation finished!")

In [ ]:
# Plotting the results
plt.figure(figsize=(10, 8), dpi=100)

colors = {
    'sarsa': '#E11D48',
    'expected_sarsa': '#2563EB',
    'q_learning': '#16A34A'
}

# Asymptotic curves
plt.plot(step_sizes, performance[0, :], label='Sarsa (Asymptotic)', color=colors['sarsa'], linestyle='-', marker='s', markersize=8)
plt.plot(step_sizes, performance[1, :], label='Expected Sarsa (Asymptotic)', color=colors['expected_sarsa'], linestyle='-', marker='o', markersize=8)
plt.plot(step_sizes, performance[2, :], label='Q-learning (Asymptotic)', color=colors['q_learning'], linestyle='-', marker='^', markersize=8)

# Interim curves
plt.plot(step_sizes, performance[3, :], label='Sarsa (Interim)', color=colors['sarsa'], linestyle='--', marker='s', markersize=8, markerfacecolor='none', alpha=0.7)
plt.plot(step_sizes, performance[4, :], label='Expected Sarsa (Interim)', color=colors['expected_sarsa'], linestyle='--', marker='o', markersize=8, markerfacecolor='none', alpha=0.7)
plt.plot(step_sizes, performance[5, :], label='Q-learning (Interim)', color=colors['q_learning'], linestyle='--', marker='^', markersize=8, markerfacecolor='none', alpha=0.7)

plt.xlabel('Step-size Parameter, alpha')
plt.ylabel('Average Reward per Episode')
plt.title('Interim and Asymptotic Performance of TD Control Methods')
plt.xlim([0.05, 1.05])
plt.ylim([-140, -10])
plt.xticks(step_sizes)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, loc='lower left')
plt.show()

## 5. Discussion & Key Takeaways

### Why Sarsa Asymptotic Performance Degrades
Sarsa is on-policy and uses the actual exploratory action $A_{t+1}$ to update the value of $Q(S_t, A_t)$. When the step-size parameter $\alpha$ is large (approaching 1.0), Sarsa's action-value updates are subject to high variance due to the exploration steps chosen by the policy. This leads to instability in the learned policy, making the agent frequently fall off the cliff during learning, causing its asymptotic performance to drop off dramatically.

### Why Q-learning Asymptotic Performance is Flat and Poor
Q-learning is off-policy and learns the optimal path directly along the edge of the cliff. In the limit, this path has a length of 13 steps (expected reward of $-13$). However, Q-learning's behavior policy during training is $\varepsilon$-greedy with $\varepsilon = 0.1$. The $10\%$ exploration rate means that at any point along the edge, the agent stands a $2.5\%$ chance of choosing the action **Down** into the cliff. Navigating all 12 steps along the cliff edge without falling is difficult, resulting in the agent falling off the cliff in $26.2\%$ of episodes. Since Q-learning updates assume the greedy path is followed, the updates do not account for these random falls, leading to a constant low average reward ($\approx -50$) during training.

### Why Expected Sarsa is Robust
Expected Sarsa combines the on-policy evaluation of Sarsa with the variance reduction of taking an expectation. By taking the expected value over all next action probabilities under the target policy, it **integrates out** action-selection variance. At high learning rates ($\alpha = 1.0$), Expected Sarsa remains completely stable and converges to the safe path, achieving an asymptotic average reward of $\approx -21$, outperforming Sarsa's best performance at any step-size.

## 6. Summary

### Q&A
* **Q**: How does Expected Sarsa reduce variance?
  * **A**: By updating the current action-value using the expected value of the next state-action pairs under the target policy rather than a sampled action value. This mathematically integrates out the variance introduced by the random selection of the next action.
* **Q**: Why does Expected Sarsa outperform Sarsa and Q-learning in Cliff Walking?
  * **A**: It avoids Sarsa's sensitivity to high learning rates by removing action-selection variance, and it avoids Q-learning's exploration penalty by accounting for the behavior policy's exploratory steps, allowing it to learn the safe path while maintaining stable convergence.
  
### Data Analysis Key Findings
* **Expected Sarsa (Asymptotic)** was highly robust, maintaining an average reward of $\approx -21.0$ even at $\alpha = 1.0$.
* **Sarsa (Asymptotic)** was sensitive to the step-size, degrading from a peak of $\approx -24.2$ at $\alpha = 0.3$ down to $-65.1$ at $\alpha = 1.0$.
* **Q-learning (Asymptotic)** remained flat at around $-50.0$ due to the $10\%$ exploration rate causing frequent falls.
* **Interim Performance** was dominated by Expected Sarsa for all step sizes, proving its sample efficiency and learning speed.

### Insights or Next Steps
* **Insight**: When using standard TD control, Expected Sarsa should be preferred over Sarsa and Q-learning if the action space is small and the behavior policy is stochastic, because it yields lower variance updates and is highly robust to hyperparameters.
* **Next Step**: Evaluate the methods in stochastic transition environments, where Expected Sarsa's variance reduction is even more beneficial.